In [ ]:
!pip install biopython gradio openpyxl pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 16.7 MB/s eta 0:00:00


In [4]:
import pandas as pd

file_path = "/content/mock_antibody_repertoire.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (30, 16)
Columns: ['sequence_id', 'species', 'chain_type', 'isotype', 'v_gene', 'd_gene', 'j_gene', 'framework1_aa', 'cdr1_aa', 'framework2_aa', 'cdr2_aa', 'framework3_aa', 'cdr3_aa', 'framework4_aa', 'source_sample', 'notes']


,sequence_id,species,chain_type,isotype,v_gene,d_gene,j_gene,framework1_aa,cdr1_aa,framework2_aa,cdr2_aa,framework3_aa,cdr3_aa,framework4_aa,source_sample,notes
0,ABR0001,canine,kappa,NaN,IGKV2-8,NaN,IGKJ4,CDVENWCTHCDQQDIDVQCWEIWC,WWPCICVF,LQFVEWLVGEWWHNEV,DWCYHSVQ,MRWRNLIGIDWLTSMRLYDETQGMFSQCDV,WMMNYSWRDD,KSDCLWRLPNAR,lymph_node,antigen-exposed
1,ABR0002,canine,lambda,NaN,IGLV2-11,NaN,IGLJ2,HLFIPPSDGRPVKFQVKQNPIFDG,FIIASWGK,LAFQVNYWMFTYCRVP,PPPESPCH,DHRGEMYCEAWFVENYADHYPFKNYNSEES,RSSLDFEMKS,GTAHTNFVATLD,lymph_node,high-confidence annotation
2,ABR0003,feline,heavy,IgG,IGHV3-7,IGHD5,IGHJ6,IYHIPIHTSNAAKSKHYNRNNDIE,ISHMHSYY,ASNDEPHSGQMDPRPD,GGFAFWRF,YYSNFVVFAAETFQHHAKHLTIWMKVQFCN,RWTQTFVFTTARGY,AFGFSYEVCMTT,bone_marrow,naive repertoire
3,ABR0004,canine,lambda,NaN,IGLV1-5,NaN,IGLJ2,KCETRVADRMYTYTHKRTVSTITK,VHRFQEPR,MDIQDHLEFNFKFRIE,PSGIGQTP,MQHNMDNAMVRRAPMTYLTDEIEDKKCGKF,QKPFVTWSMD,KCGQDKADKDYI,PBMC,high-confidence annotation
4,ABR0005,feline,heavy,IgG,IGHV2-5,NaN,IGHJ4,YFCTIEGKCGHLLTHLRTGKNAKC,AATVHTSI,REQSVPTLHIMHFPNC,FADKQGCD,PTLYILCRGGKRAKNMVMICLHNGAMPDSK,THITADKDFPWCPA,LLIDWTFYPMSF,lymph_node,antigen-exposed


In [9]:
df.columns = df.columns.str.replace('_aa', '', regex=False)
df.head()

,sequence_id,species,chain_type,isotype,v_gene,d_gene,j_gene,framework1,cdr1,framework2,cdr2,framework3,cdr3,framework4,source_sample,notes
0,ABR0001,canine,kappa,NaN,IGKV2-8,NaN,IGKJ4,CDVENWCTHCDQQDIDVQCWEIWC,WWPCICVF,LQFVEWLVGEWWHNEV,DWCYHSVQ,MRWRNLIGIDWLTSMRLYDETQGMFSQCDV,WMMNYSWRDD,KSDCLWRLPNAR,lymph_node,antigen-exposed
1,ABR0002,canine,lambda,NaN,IGLV2-11,NaN,IGLJ2,HLFIPPSDGRPVKFQVKQNPIFDG,FIIASWGK,LAFQVNYWMFTYCRVP,PPPESPCH,DHRGEMYCEAWFVENYADHYPFKNYNSEES,RSSLDFEMKS,GTAHTNFVATLD,lymph_node,high-confidence annotation
2,ABR0003,feline,heavy,IgG,IGHV3-7,IGHD5,IGHJ6,IYHIPIHTSNAAKSKHYNRNNDIE,ISHMHSYY,ASNDEPHSGQMDPRPD,GGFAFWRF,YYSNFVVFAAETFQHHAKHLTIWMKVQFCN,RWTQTFVFTTARGY,AFGFSYEVCMTT,bone_marrow,naive repertoire
3,ABR0004,canine,lambda,NaN,IGLV1-5,NaN,IGLJ2,KCETRVADRMYTYTHKRTVSTITK,VHRFQEPR,MDIQDHLEFNFKFRIE,PSGIGQTP,MQHNMDNAMVRRAPMTYLTDEIEDKKCGKF,QKPFVTWSMD,KCGQDKADKDYI,PBMC,high-confidence annotation
4,ABR0005,feline,heavy,IgG,IGHV2-5,NaN,IGHJ4,YFCTIEGKCGHLLTHLRTGKNAKC,AATVHTSI,REQSVPTLHIMHFPNC,FADKQGCD,PTLYILCRGGKRAKNMVMICLHNGAMPDSK,THITADKDFPWCPA,LLIDWTFYPMSF,lymph_node,antigen-exposed


In [10]:
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
print(df.columns.tolist())

['sequence_id', 'species', 'chain_type', 'isotype', 'v_gene', 'd_gene', 'j_gene', 'framework1', 'cdr1', 'framework2', 'cdr2', 'framework3', 'cdr3', 'framework4', 'source_sample', 'notes']


In [11]:
required_cols = [
    "sequence_id", "species", "chain_type",
    "v_gene", "d_gene", "j_gene",
    "framework1", "cdr1", "framework2", "cdr2",
    "framework3", "cdr3", "framework4"
]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    print("Missing columns:", missing)
else:
    print("All required columns present ✅")

All required columns present ✅


In [12]:
df["full_variable_region"] = (
    df["framework1"].fillna("") +
    df["cdr1"].fillna("") +
    df["framework2"].fillna("") +
    df["cdr2"].fillna("") +
    df["framework3"].fillna("") +
    df["cdr3"].fillna("") +
    df["framework4"].fillna("")
)

df.head()

,sequence_id,species,chain_type,isotype,v_gene,d_gene,j_gene,framework1,cdr1,framework2,cdr2,framework3,cdr3,framework4,source_sample,notes,full_variable_region
0,ABR0001,canine,kappa,NaN,IGKV2-8,NaN,IGKJ4,CDVENWCTHCDQQDIDVQCWEIWC,WWPCICVF,LQFVEWLVGEWWHNEV,DWCYHSVQ,MRWRNLIGIDWLTSMRLYDETQGMFSQCDV,WMMNYSWRDD,KSDCLWRLPNAR,lymph_node,antigen-exposed,CDVENWCTHCDQQDIDVQCWEIWCWWPCICVFLQFVEWLVGEWWHN...
1,ABR0002,canine,lambda,NaN,IGLV2-11,NaN,IGLJ2,HLFIPPSDGRPVKFQVKQNPIFDG,FIIASWGK,LAFQVNYWMFTYCRVP,PPPESPCH,DHRGEMYCEAWFVENYADHYPFKNYNSEES,RSSLDFEMKS,GTAHTNFVATLD,lymph_node,high-confidence annotation,HLFIPPSDGRPVKFQVKQNPIFDGFIIASWGKLAFQVNYWMFTYCR...
2,ABR0003,feline,heavy,IgG,IGHV3-7,IGHD5,IGHJ6,IYHIPIHTSNAAKSKHYNRNNDIE,ISHMHSYY,ASNDEPHSGQMDPRPD,GGFAFWRF,YYSNFVVFAAETFQHHAKHLTIWMKVQFCN,RWTQTFVFTTARGY,AFGFSYEVCMTT,bone_marrow,naive repertoire,IYHIPIHTSNAAKSKHYNRNNDIEISHMHSYYASNDEPHSGQMDPR...
3,ABR0004,canine,lambda,NaN,IGLV1-5,NaN,IGLJ2,KCETRVADRMYTYTHKRTVSTITK,VHRFQEPR,MDIQDHLEFNFKFRIE,PSGIGQTP,MQHNMDNAMVRRAPMTYLTDEIEDKKCGKF,QKPFVTWSMD,KCGQDKADKDYI,PBMC,high-confidence annotation,KCETRVADRMYTYTHKRTVSTITKVHRFQEPRMDIQDHLEFNFKFR...
4,ABR0005,feline,heavy,IgG,IGHV2-5,NaN,IGHJ4,YFCTIEGKCGHLLTHLRTGKNAKC,AATVHTSI,REQSVPTLHIMHFPNC,FADKQGCD,PTLYILCRGGKRAKNMVMICLHNGAMPDSK,THITADKDFPWCPA,LLIDWTFYPMSF,lymph_node,antigen-exposed,YFCTIEGKCGHLLTHLRTGKNAKCAATVHTSIREQSVPTLHIMHFP...


In [13]:
import sqlite3

conn = sqlite3.connect("antibody_repertoire.db")

df.to_sql("antibodies", conn, if_exists="replace", index=False)

print("Database created successfully ✅")

Database created successfully ✅


In [14]:
def search_sequences(region="cdr3", motif="", species=None, chain_type=None):
    query = f"SELECT * FROM antibodies WHERE {region} LIKE ?"
    params = [f"%{motif}%"]

    if species:
        query += " AND lower(species) = ?"
        params.append(species.lower())

    if chain_type:
        query += " AND lower(chain_type) = ?"
        params.append(chain_type.lower())

    return pd.read_sql_query(query, conn, params=params)

In [15]:
search_sequences(region="cdr3", motif="YY")

,sequence_id,species,chain_type,isotype,v_gene,d_gene,j_gene,framework1,cdr1,framework2,cdr2,framework3,cdr3,framework4,source_sample,notes,full_variable_region
0,ABR0018,equine,heavy,IgG,IGHV2-5,IGHD3,IGHJ4,VKLGQCMAQWWCSWTCEQWPRDAP,YWFSQVED,SHFAQAAEDHEFSAKW,IRGCNFDL,VSRKCCACAYDPLLYGSYCMNWRSGFENGQ,SPRKWMLKCYYMYA,FYLWQIPPPYIR,lymph_node,naive repertoire,VKLGQCMAQWWCSWTCEQWPRDAPYWFSQVEDSHFAQAAEDHEFSA...


In [16]:
def advanced_search(
    species=None,
    chain_type=None,
    v_gene=None,
    j_gene=None,
    region=None,
    motif=None
):
    query = "SELECT * FROM antibodies WHERE 1=1"
    params = []

    if species:
        query += " AND lower(species) = ?"
        params.append(species.lower())

    if chain_type:
        query += " AND lower(chain_type) = ?"
        params.append(chain_type.lower())

    if v_gene:
        query += " AND v_gene LIKE ?"
        params.append(f"%{v_gene}%")

    if j_gene:
        query += " AND j_gene LIKE ?"
        params.append(f"%{j_gene}%")

    if region and motif:
        query += f" AND {region} LIKE ?"
        params.append(f"%{motif}%")

    return pd.read_sql_query(query, conn, params=params)

In [17]:
advanced_search(species="canine", region="cdr3", motif="GG")

,sequence_id,species,chain_type,isotype,v_gene,d_gene,j_gene,framework1,cdr1,framework2,cdr2,framework3,cdr3,framework4,source_sample,notes,full_variable_region
0,ABR0021,canine,heavy,IgM,IGHV2-5,None,IGHJ2,CLFARTMTFRATLGNQCQHKWGFG,TIGHYDDY,SKGHFYHWLHADTQCT,NMLSDAQS,FKIGWNCGNWYANTRTDENIMPWCLESRTA,TVFAIDIYGGELKV,AAEHKAYWRTIR,PBMC,high-confidence annotation,CLFARTMTFRATLGNQCQHKWGFGTIGHYDDYSKGHFYHWLHADTQ...


In [18]:
def get_sequence_by_id(sequence_id):
    query = "SELECT * FROM antibodies WHERE sequence_id = ?"
    return pd.read_sql_query(query, conn, params=[sequence_id])

In [20]:
get_sequence_by_id("ABR0013")

,sequence_id,species,chain_type,isotype,v_gene,d_gene,j_gene,framework1,cdr1,framework2,cdr2,framework3,cdr3,framework4,source_sample,notes,full_variable_region
0,ABR0013,feline,heavy,IgA,IGHV2-5,IGHD3,IGHJ2,AIFREDFKPKACVNYWRYTSIGAC,CVAPGIGC,EAYVHFQHTYTQYGTL,DLCSVAPQ,RDRGIEKICEMKCKVQTKLHDTAGKIHGMH,PMYIPVSSTAAQIW,LHPYWDWGFCAE,PBMC,antigen-exposed,AIFREDFKPKACVNYWRYTSIGACCVAPGIGCEAYVHFQHTYTQYG...


In [22]:
!pip install biopython gradio pandas
from Bio import pairwise2
from Bio.pairwise2 import format_alignment

def align_regions(seq_id1, seq_id2, region="cdr3"):
    df1 = get_sequence_by_id(seq_id1)
    df2 = get_sequence_by_id(seq_id2)

    if df1.empty or df2.empty:
        return "Sequence not found"

    seq1 = str(df1.iloc[0][region])
    seq2 = str(df2.iloc[0][region])

    alignments = pairwise2.align.globalxx(seq1, seq2)

    if not alignments:
        return "No alignment"

    return format_alignment(*alignments[0])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 17.2 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [24]:
print(align_regions("ABR0005", "ABR0010", region="cdr3"))

THITADKDFPW-C------PA----
       |    |      |     
-------D---HCSVSDQEP-VFVD
  Score=3



In [25]:
def alignment_score(seq_id1, seq_id2, region="cdr3"):
    df1 = get_sequence_by_id(seq_id1)
    df2 = get_sequence_by_id(seq_id2)

    if df1.empty or df2.empty:
        return None

    seq1 = str(df1.iloc[0][region])
    seq2 = str(df2.iloc[0][region])

    alignments = pairwise2.align.globalxx(seq1, seq2)

    if not alignments:
        return None

    return alignments[0].score

In [26]:
def search_all_regions(motif, species=None, chain_type=None):
    regions = [
        "framework1", "cdr1", "framework2",
        "cdr2", "framework3", "cdr3", "framework4"
    ]

    query = "SELECT * FROM antibodies WHERE ("
    query += " OR ".join([f"{r} LIKE ?" for r in regions])
    query += ")"

    params = [f"%{motif}%" for _ in regions]

    if species:
        query += " AND lower(species) = ?"
        params.append(species.lower())

    if chain_type:
        query += " AND lower(chain_type) = ?"
        params.append(chain_type.lower())

    return pd.read_sql_query(query, conn, params=params)

In [28]:
search_all_regions("AAT")

,sequence_id,species,chain_type,isotype,v_gene,d_gene,j_gene,framework1,cdr1,framework2,cdr2,framework3,cdr3,framework4,source_sample,notes,full_variable_region
0,ABR0005,feline,heavy,IgG,IGHV2-5,None,IGHJ4,YFCTIEGKCGHLLTHLRTGKNAKC,AATVHTSI,REQSVPTLHIMHFPNC,FADKQGCD,PTLYILCRGGKRAKNMVMICLHNGAMPDSK,THITADKDFPWCPA,LLIDWTFYPMSF,lymph_node,antigen-exposed,YFCTIEGKCGHLLTHLRTGKNAKCAATVHTSIREQSVPTLHIMHFP...


In [29]:
from google.colab import files
files.download("antibody_repertoire.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
DB_PATH = "antibody_repertoire.db"

In [32]:
import sqlite3

DB_PATH = "antibody_repertoire.db"
conn = sqlite3.connect(DB_PATH)
df.to_sql("antibodies", conn, if_exists="replace", index=False)
conn.close()

print("Database created successfully.")

Database created successfully.


In [33]:
import pandas as pd
import sqlite3

ALLOWED_REGIONS = [
    "framework1", "cdr1", "framework2", "cdr2",
    "framework3", "cdr3", "framework4", "full_variable_region"
]

def search_sequences(region="cdr3", motif="", species=None, chain_type=None):
    if region not in ALLOWED_REGIONS:
        return pd.DataFrame({"error": [f"Invalid region: {region}"]})

    query = f"SELECT * FROM antibodies WHERE {region} LIKE ?"
    params = [f"%{motif}%"]

    if species:
        query += " AND lower(species) = ?"
        params.append(species.lower())

    if chain_type:
        query += " AND lower(chain_type) = ?"
        params.append(chain_type.lower())

    with sqlite3.connect(DB_PATH) as conn:
        result = pd.read_sql_query(query, conn, params=params)

    return result


def get_sequence_by_id(sequence_id):
    query = "SELECT * FROM antibodies WHERE sequence_id = ?"
    with sqlite3.connect(DB_PATH) as conn:
        result = pd.read_sql_query(query, conn, params=[sequence_id])
    return result

In [34]:
from Bio import pairwise2
from Bio.pairwise2 import format_alignment

def align_regions(seq_id1, seq_id2, region="cdr3"):
    if region not in ALLOWED_REGIONS:
        return f"Invalid region: {region}"

    df1 = get_sequence_by_id(seq_id1)
    df2 = get_sequence_by_id(seq_id2)

    if df1.empty:
        return f"{seq_id1} not found"
    if df2.empty:
        return f"{seq_id2} not found"

    seq1 = str(df1.iloc[0][region]).strip()
    seq2 = str(df2.iloc[0][region]).strip()

    if not seq1 or seq1.lower() == "nan":
        return f"{seq_id1} has no sequence in {region}"
    if not seq2 or seq2.lower() == "nan":
        return f"{seq_id2} has no sequence in {region}"

    alignments = pairwise2.align.globalxx(seq1, seq2)

    if not alignments:
        return "No alignment found"

    best = alignments[0]
    return format_alignment(*best)

In [35]:
def gradio_search(region, motif, species, chain_type):
    try:
        species = species.strip() if species else None
        chain_type = chain_type.strip() if chain_type else None
        motif = motif.strip() if motif else ""

        if motif == "":
            return pd.DataFrame()

        return search_sequences(
            region=region,
            motif=motif,
            species=species,
            chain_type=chain_type
        )
    except Exception as e:
        return pd.DataFrame({"error": [str(e)]})


def gradio_align(seq_id1, seq_id2, region):
    try:
        seq_id1 = seq_id1.strip()
        seq_id2 = seq_id2.strip()
        return align_regions(seq_id1, seq_id2, region)
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
import gradio as gr

search_regions = [
    "framework1", "cdr1", "framework2", "cdr2",
    "framework3", "cdr3", "framework4", "full_variable_region"
]

with gr.Blocks() as demo:
    gr.Markdown("# Antibody Repertoire Database Prototype")

    with gr.Tab("Search"):
        region = gr.Dropdown(search_regions, value="cdr3", label="Region")
        motif = gr.Textbox(label="Motif / partial sequence")
        species = gr.Textbox(label="Species (optional)")
        chain_type = gr.Textbox(label="Chain type (optional)")
        search_btn = gr.Button("Search")
        search_output = gr.Dataframe()

        search_btn.click(
            fn=gradio_search,
            inputs=[region, motif, species, chain_type],
            outputs=search_output
        )

    with gr.Tab("Align"):
        seq_id1 = gr.Textbox(label="Sequence ID 1")
        seq_id2 = gr.Textbox(label="Sequence ID 2")
        align_region = gr.Dropdown(search_regions[:-1], value="cdr3", label="Region")
        align_btn = gr.Button("Align")
        align_output = gr.Textbox(label="Alignment", lines=15)

        align_btn.click(
            fn=gradio_align,
            inputs=[seq_id1, seq_id2, align_region],
            outputs=align_output
        )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d52917e902df86429d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
